# REGIME-SHIFT — Macro-Aware Tactical Asset Allocation Engine



This notebook runs a complete, bias-aware macro/tactical allocation pipeline:



- **Inputs**: daily prices for `SPY` (equities), `TLT` (rates), `GLD` (safe haven), `^VIX` (risk proxy) + optional FRED macro.

- **Regimes**: a 3-state Gaussian HMM fit on features (asset returns + VIX + optional macro).

- **Allocation**: convex optimizer with objectives/constraints mapped to the inferred regime.

- **Validation**: strict walk-forward refits (only past data), monthly rebalances, explicit turnover cost.



Outputs are written to `outputs/` (charts + CSVs).

## Setup / Repro



- Install deps: `pip install -r requirements.txt` then `pip install -e .`

- Optional macro (FRED): set `FRED_API_KEY` in your environment (or `.env` and load it in your shell).



**Anti-lookahead rule** used here: weights decided at rebalance date $t$ are applied starting the next trading day ($t+1$). Transaction cost is charged on that first holding day.

## Configuration



Key knobs:



- `train_years`: rolling history used to refit the HMM + estimate moments for optimization

- `n_states`: HMM hidden states (3 → Bull/Bear/Crisis after labeling)

- `rebalance`: rebalance schedule (`ME` = month-end; any `pandas.date_range` frequency works)

- `cost_bps`: transaction friction applied as `cost_bps/10000 * turnover` per rebalance

In [ ]:
from regime_shift.config import BacktestConfig, OptimizerConfig, Universe
from regime_shift.pipeline import run_engine

bt = BacktestConfig(start="2006-01-01", end="2025-12-31", train_years=3, n_states=3, rebalance="ME", cost_bps=7.5)
opt = OptimizerConfig()
universe = Universe(risky="SPY", rates="TLT", haven="GLD", vix="^VIX")

## Run



`run_engine(...)` will:



1. Download daily prices from Yahoo Finance

2. Optionally load FRED macro (if `FRED_API_KEY` is set)

3. Build features and fit the HMM walk-forward

4. Optimize weights per regime at each rebalance

5. Backtest with explicit turnover costs

6. Save plots/CSVs to `outputs/`

In [ ]:
res = run_engine(bt=bt, opt=opt, universe=universe, out_dir="outputs")
res.metrics.round(4)

## Results: Performance vs Baselines



The table below compares:



- **Dynamic**: HMM-driven regime allocation with transaction costs

- **60/40**: static SPY/TLT (GLD = 0)

- **Equal**: equal-weight across available assets



Metrics: CAGR, Sharpe, Sortino, Max Drawdown, Calmar, and (for Dynamic) average turnover.

## HMM Transition Probabilities



This prints the fitted state transition matrix (rows → next-state probabilities).



Regime labels are assigned *after fitting* by scoring each hidden state on equity (`SPY`) mean return and volatility:



- **Bull**: highest equity mean, lower volatility

- **Crisis**: lowest equity mean, highest volatility

- **Bear**: the remaining state

In [ ]:
res.transition

## Equity Curve



The plot below shows the equity curves (growth of $1) for Dynamic vs baselines over the chosen window.

In [ ]:
ax = res.curves.plot(figsize=(11,5), title="Equity Curves", grid=True)
ax.set_ylabel("Growth of $1")

## Weights and Turnover



- `weights`: optimizer output at each rebalance date

- `turnover`: $\sum_i |w_{t,i} - w_{t-1,i}|$ (drives transaction costs)



Cost model: on each rebalance, cost is `cost_bps/10000 * turnover` and is subtracted from the first daily return after the rebalance.

In [ ]:
res.weights.tail()

In [ ]:
res.turnover.describe()